### Magnitude Model

The magnitude model predicts the expected absolute overnight return:

\[
\text{actual\_magnitude\_pct} = |\text{actual\_return\_pct}|
\]

The model ignores direction and estimates only the expected size of the next overnight move.

Workflow:

1. Load the processed modelling panel.
2. Preserve the existing chronological train, validation, embargo and test splits.
3. Establish trailing-magnitude baselines.
4. Train a pooled LightGBM regressor with symbol as a categorical feature.
5. Evaluate magnitude score, MAE, RMSE and cross-sectional rank IC.
6. Select the final model using validation data only.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

matches = list(
    PROJECT_ROOT.rglob("magnitude_model_panel.parquet")
)

print("\nMatches found:")
for path in matches:
    print(path)

In [ ]:
matches = list(
    PROJECT_ROOT.rglob("magnitude_model_panel.parquet")
)

assert len(matches) == 1, (
    f"Expected exactly one magnitude_model_panel.parquet, found {len(matches)}: "
    f"{matches}"
)

MAGNITUDE_PANEL_PATH = matches[0]

model_df = pd.read_parquet(
    MAGNITUDE_PANEL_PATH
)

model_df["pred_date"] = pd.to_datetime(
    model_df["pred_date"]
)

if "target_date" in model_df.columns:
    model_df["target_date"] = pd.to_datetime(
        model_df["target_date"]
    )

model_df = (
    model_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

print("Loaded:", MAGNITUDE_PANEL_PATH)
print("Shape:", model_df.shape)
print("Symbols:", model_df["symbol"].nunique())
print(model_df["split"].value_counts(dropna=False))

In [ ]:
required_columns = [
    "symbol",
    "pred_date",
    "split",
    "actual_return_pct",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in model_df.columns
]

assert not missing_required_columns, (
    f"Missing required columns: {missing_required_columns}"
)

assert not model_df.duplicated(
    ["symbol", "pred_date"]
).any()

assert model_df["symbol"].nunique() == 208

assert model_df["split"].isin(
    ["train", "valid", "test", "embargo"]
).all()

assert model_df["actual_return_pct"].notna().all()

print("Magnitude panel checks passed.")

In [ ]:
model_df["actual_magnitude_pct"] = (
    model_df["actual_return_pct"].abs()
)

model_df["log_actual_magnitude"] = np.log1p(
    model_df["actual_magnitude_pct"]
)

assert model_df["actual_magnitude_pct"].ge(0).all()

model_df[
    "actual_magnitude_pct"
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
model_df["trailing_magnitude_20d"] = (
    model_df
    .groupby("symbol")["actual_magnitude_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(
                window=20,
                min_periods=20,
            )
            .mean()
        )
    )
)

assert (
    model_df["trailing_magnitude_20d"]
    .dropna()
    .ge(0)
    .all()
)

print(
    model_df[
        "trailing_magnitude_20d"
    ].isna().sum()
)

model_df[
    [
        "actual_magnitude_pct",
        "trailing_magnitude_20d",
    ]
].describe().T

In [ ]:
train_df = model_df[
    model_df["split"] == "train"
].copy()

valid_df = model_df[
    model_df["split"] == "valid"
].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(valid_df))
print(
    "Embargo rows excluded:",
    model_df["split"].eq("embargo").sum(),
)
print(
    "Test rows untouched:",
    model_df["split"].eq("test").sum(),
)

In [ ]:
explicitly_excluded_columns = {
    # Identifiers and split metadata
    "pred_date",
    "target_date",
    "split",

    # Targets and realised outcomes
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
    "log_actual_magnitude",
    "universe_mean_pct",

    # Possible future-derived fields
    "next_open",
    "next_open_price",
    "target_return",
    "target_direction",
    "target_magnitude",
}

suspicious_target_terms = [
    "actual_",
    "target_",
    "next_open",
    "future_",
    "forward_",
    "label",
]

magnitude_feature_columns = []

for column in model_df.columns:
    if column in explicitly_excluded_columns:
        continue

    if column == "symbol":
        magnitude_feature_columns.append(column)
        continue

    if any(
        term in column.lower()
        for term in suspicious_target_terms
    ):
        continue

    if pd.api.types.is_numeric_dtype(
        model_df[column]
    ):
        magnitude_feature_columns.append(column)

magnitude_feature_columns = list(
    dict.fromkeys(magnitude_feature_columns)
)

assert "symbol" in magnitude_feature_columns
assert "actual_return_pct" not in magnitude_feature_columns
assert "actual_magnitude_pct" not in magnitude_feature_columns
assert "log_actual_magnitude" not in magnitude_feature_columns

print(
    "Magnitude feature count:",
    len(magnitude_feature_columns),
)

print("\nMagnitude features:")
for feature in magnitude_feature_columns:
    print("-", feature)

In [ ]:
X_train = train_df[
    magnitude_feature_columns
].copy()

X_valid = valid_df[
    magnitude_feature_columns
].copy()

y_train_log = train_df[
    "log_actual_magnitude"
].copy()

y_valid_pct = valid_df[
    "actual_magnitude_pct"
].copy()

for frame in [
    X_train,
    X_valid,
]:
    frame["symbol"] = frame[
        "symbol"
    ].astype("category")

numeric_features = [
    column
    for column in magnitude_feature_columns
    if column != "symbol"
]

assert not np.isinf(
    X_train[
        numeric_features
    ].to_numpy(dtype=float)
).any()

assert not np.isinf(
    X_valid[
        numeric_features
    ].to_numpy(dtype=float)
).any()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Feature matrices passed validation.")

In [ ]:
def evaluate_magnitude_predictions(
    evaluation_df: pd.DataFrame,
    predictions: np.ndarray,
    baseline_predictions: np.ndarray,
) -> dict:
    actual = evaluation_df[
        "actual_magnitude_pct"
    ].to_numpy(dtype=float)

    predictions = np.asarray(
        predictions,
        dtype=float,
    )

    baseline_predictions = np.asarray(
        baseline_predictions,
        dtype=float,
    )

    predictions = np.clip(
        predictions,
        0,
        None,
    )

    valid_mask = (
        np.isfinite(actual)
        & np.isfinite(predictions)
        & np.isfinite(baseline_predictions)
    )

    actual = actual[valid_mask]
    predictions = predictions[valid_mask]
    baseline_predictions = baseline_predictions[
        valid_mask
    ]

    absolute_error = np.abs(
        predictions - actual
    )

    denominator = np.sum(actual)

    magnitude_score = (
        1
        - np.sum(absolute_error) / denominator
        if denominator > 0
        else np.nan
    )

    mae = mean_absolute_error(
        actual,
        predictions,
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predictions,
        )
    )

    baseline_sse = np.sum(
        (
            baseline_predictions - actual
        ) ** 2
    )

    model_sse = np.sum(
        (
            predictions - actual
        ) ** 2
    )

    r2_vs_vol = (
        1 - model_sse / baseline_sse
        if baseline_sse > 0
        else np.nan
    )

    rank_ics = []

    rank_ic_frame = evaluation_df.loc[
        valid_mask,
        ["pred_date"],
    ].copy()

    rank_ic_frame["actual"] = actual
    rank_ic_frame["prediction"] = predictions

    for _, day in rank_ic_frame.groupby(
        "pred_date"
    ):
        if (
            len(day) >= 3
            and day["actual"].nunique() > 1
            and day["prediction"].nunique() > 1
        ):
            correlation = spearmanr(
                day["prediction"],
                day["actual"],
            ).statistic

            if np.isfinite(correlation):
                rank_ics.append(correlation)

    rank_ics = np.asarray(
        rank_ics,
        dtype=float,
    )

    rank_ic = (
        float(rank_ics.mean())
        if len(rank_ics)
        else np.nan
    )

    if (
        len(rank_ics) > 1
        and rank_ics.std(ddof=1) > 0
    ):
        rank_ic_t = (
            rank_ic
            / (
                rank_ics.std(ddof=1)
                / np.sqrt(len(rank_ics))
            )
        )
    else:
        rank_ic_t = np.nan

    return {
        "magnitude_score": magnitude_score,
        "mae": mae,
        "rmse": rmse,
        "rank_ic": rank_ic,
        "rank_ic_t": rank_ic_t,
        "r2_vs_vol": r2_vs_vol,
        "n_obs": int(len(actual)),
        "n_rank_ic_days": int(len(rank_ics)),
    }

In [ ]:
valid_baseline_mask = valid_df[
    "trailing_magnitude_20d"
].notna()

valid_baseline_df = valid_df.loc[
    valid_baseline_mask
].copy()

valid_baseline_predictions = valid_baseline_df[
    "trailing_magnitude_20d"
].to_numpy()

baseline_metrics = evaluate_magnitude_predictions(
    evaluation_df=valid_baseline_df,
    predictions=valid_baseline_predictions,
    baseline_predictions=valid_baseline_predictions,
)

pd.Series(
    baseline_metrics,
    name="trailing_20d_baseline",
)

In [ ]:
magnitude_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1500,
    num_leaves=31,
    learning_rate=0.03,
    min_child_samples=100,
    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

magnitude_model.fit(
    X_train,
    y_train_log,
    categorical_feature=["symbol"],
    eval_set=[
        (
            X_valid,
            np.log1p(
                valid_df[
                    "actual_magnitude_pct"
                ]
            ),
        )
    ],
    eval_metric="l1",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True,
        )
    ],
)

print(
    "Best iteration:",
    magnitude_model.best_iteration_,
)

In [ ]:
large_tree_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=1500,
    num_leaves=63,
    learning_rate=0.03,
    min_child_samples=50,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

large_tree_model.fit(
    X_train,
    y_train_log,
    categorical_feature=["symbol"],
    eval_set=[
        (
            X_valid,
            np.log1p(
                valid_df[
                    "actual_magnitude_pct"
                ]
            ),
        )
    ],
    eval_metric="l1",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True,
        )
    ],
)

large_tree_prediction_log = (
    large_tree_model.predict(
        X_valid,
        num_iteration=(
            large_tree_model.best_iteration_
        ),
    )
)

large_tree_pred_magnitude_pct = np.expm1(
    large_tree_prediction_log
)

large_tree_model_predictions = (
    large_tree_pred_magnitude_pct[
        valid_evaluation_mask.to_numpy()
    ]
)

large_tree_metrics = (
    evaluate_magnitude_predictions(
        evaluation_df=valid_evaluation_df,
        predictions=large_tree_model_predictions,
        baseline_predictions=(
            valid_reference_predictions
        ),
    )
)

print(
    "Large-tree best iteration:",
    large_tree_model.best_iteration_,
)

pd.Series(
    large_tree_metrics,
    name="large_tree_l1",
)

In [ ]:
magnitude_iteration_comparison = pd.DataFrame(
    [
        {
            "model": "trailing_20d_baseline",
            **baseline_metrics,
        },
        {
            "model": "current_l1",
            **lightgbm_metrics,
        },
        {
            "model": "huber",
            **huber_metrics,
        },
        {
            "model": "large_tree_l1",
            **large_tree_metrics,
        },
    ]
)

magnitude_iteration_comparison

In [ ]:
valid_prediction_log = (
    magnitude_model.predict(
        X_valid,
        num_iteration=(
            magnitude_model.best_iteration_
        ),
    )
)

valid_pred_magnitude_pct = np.expm1(
    valid_prediction_log
)

valid_pred_magnitude_pct = np.clip(
    valid_pred_magnitude_pct,
    0,
    None,
)

pd.Series(
    valid_pred_magnitude_pct
).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
valid_evaluation_mask = valid_df[
    "trailing_magnitude_20d"
].notna()

valid_evaluation_df = valid_df.loc[
    valid_evaluation_mask
].copy()

valid_model_predictions = (
    valid_pred_magnitude_pct[
        valid_evaluation_mask.to_numpy()
    ]
)

valid_reference_predictions = (
    valid_evaluation_df[
        "trailing_magnitude_20d"
    ].to_numpy()
)

lightgbm_metrics = evaluate_magnitude_predictions(
    evaluation_df=valid_evaluation_df,
    predictions=valid_model_predictions,
    baseline_predictions=valid_reference_predictions,
)

magnitude_model_comparison = pd.DataFrame(
    [
        {
            "model": "trailing_20d_baseline",
            **baseline_metrics,
        },
        {
            "model": "lightgbm_log_magnitude",
            **lightgbm_metrics,
        },
    ]
)

magnitude_model_comparison

In [ ]:
prediction_diagnostics = pd.DataFrame(
    {
        "actual_magnitude_pct": (
            valid_evaluation_df[
                "actual_magnitude_pct"
            ].to_numpy()
        ),
        "baseline_magnitude_pct": (
            valid_reference_predictions
        ),
        "pred_magnitude_pct": (
            valid_model_predictions
        ),
    }
)

prediction_diagnostics.describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
).T

In [ ]:
magnitude_feature_importance = pd.DataFrame(
    {
        "feature": magnitude_feature_columns,
        "gain_importance": (
            magnitude_model.booster_
            .feature_importance(
                importance_type="gain"
            )
        ),
        "split_importance": (
            magnitude_model.booster_
            .feature_importance(
                importance_type="split"
            )
        ),
    }
).sort_values(
    "gain_importance",
    ascending=False,
).reset_index(drop=True)

magnitude_feature_importance.head(25)

In [ ]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

magnitude_validation_predictions = (
    valid_df[
        [
            "symbol",
            "pred_date",
            "actual_return_pct",
            "actual_magnitude_pct",
            "trailing_magnitude_20d",
        ]
    ]
    .copy()
)

magnitude_validation_predictions[
    "pred_magnitude_pct"
] = valid_pred_magnitude_pct

MAGNITUDE_VALIDATION_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_validation_predictions.parquet"
)

magnitude_validation_predictions.to_parquet(
    MAGNITUDE_VALIDATION_PREDICTIONS_PATH,
    index=False,
)

print(
    "Saved:",
    MAGNITUDE_VALIDATION_PREDICTIONS_PATH.resolve(),
)

print(
    "Exists:",
    MAGNITUDE_VALIDATION_PREDICTIONS_PATH.exists(),
)

In [ ]:
MAGNITUDE_RESULTS_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_model_comparison.csv"
)

MAGNITUDE_IMPORTANCE_PATH = (
    PROCESSED_DATA_DIR
    / "magnitude_feature_importance.csv"
)

magnitude_model_comparison.to_csv(
    MAGNITUDE_RESULTS_PATH,
    index=False,
)

magnitude_feature_importance.to_csv(
    MAGNITUDE_IMPORTANCE_PATH,
    index=False,
)

print("Saved:", MAGNITUDE_RESULTS_PATH)
print("Saved:", MAGNITUDE_IMPORTANCE_PATH)

In [ ]:
FINAL_MAGNITUDE_CONFIG_PATH = (
    PROCESSED_DATA_DIR
    / "final_magnitude_model_config.json"
)

final_magnitude_config = {
    "model_family": "LightGBM regressor",
    "target": "log1p(actual_magnitude_pct)",
    "prediction_output": "expm1(raw_prediction)",
    "objective": "regression_l1",
    "best_iteration": int(
        magnitude_model.best_iteration_
    ),
    "validation_metrics": {
        key: (
            float(value)
            if isinstance(
                value,
                (float, np.floating),
            )
            else int(value)
        )
        for key, value in (
            lightgbm_metrics.items()
        )
    },
    "hyperparameters": {
        "num_leaves": 31,
        "learning_rate": 0.03,
        "min_child_samples": 100,
        "feature_fraction": 0.70,
        "bagging_fraction": 0.80,
        "reg_lambda": 1.0,
        "random_state": SEED,
    },
    "categorical_features": ["symbol"],
    "selection_split": "valid",
    "test_used_for_selection": False,
}

with open(
    FINAL_MAGNITUDE_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_magnitude_config,
        file,
        indent=2,
    )

print("Saved:", FINAL_MAGNITUDE_CONFIG_PATH)

In [ ]:
print("Magnitude notebook complete.")
print()

print("Baseline validation metrics:")
print(pd.Series(baseline_metrics))
print()

print("LightGBM validation metrics:")
print(pd.Series(lightgbm_metrics))
print()

print("Test split used for selection: False")

#### Final Magnitude Model Decision

The pooled LightGBM regressor is retained as the final magnitude model.

Validation performance:

- Magnitude score: 0.254340
- MAE: 0.465360%
- RMSE: 0.926578%
- Cross-sectional rank IC: 0.245931
- Rank IC t-statistic: 32.431730
- R² versus trailing 20-session magnitude baseline: 0.054120

Reconciliation note:

- The reproducible Notebook 04 benchmark uses 38 features including `symbol`.
- The final production panel exposes 8 additional rolling-beta helper columns, which is why production logging can show 46 magnitude features without changing this notebook-equivalent benchmark.
- If the production benchmark is run on the direction warm-up filtered rows instead of the full magnitude validation split, the validation metrics move, but that is a different evaluation path rather than a stale Notebook 04 reference.

The model materially outperformed the trailing 20-session stock-level baseline across
magnitude score, absolute error, squared error and cross-sectional ranking.

The test split was not used for model selection.

In [ ]:
huber_model = lgb.LGBMRegressor(
    objective="huber",
    alpha=0.90,

    learning_rate=0.03,
    n_estimators=5000,

    num_leaves=31,
    min_child_samples=100,

    feature_fraction=0.70,
    bagging_fraction=0.80,
    bagging_freq=1,

    random_state=SEED,
)

huber_model.fit(
    X_train,
    y_train_log,

    eval_set=[
        (X_valid, np.log1p(y_valid_pct))
    ],

    eval_metric="l1",

    callbacks=[
        lgb.early_stopping(75),
        lgb.log_evaluation(0),
    ],
)

huber_prediction_log = huber_model.predict(
    X_valid,
    num_iteration=huber_model.best_iteration_,
)

huber_pred_magnitude_pct = np.expm1(
    huber_prediction_log
)


huber_pred_magnitude_pct = huber_pred_magnitude_pct[
    valid_evaluation_mask
]

huber_metrics = evaluate_magnitude_predictions(
    evaluation_df=valid_evaluation_df,
    predictions=huber_pred_magnitude_pct,
    baseline_predictions=valid_reference_predictions,
)

comparison = pd.DataFrame(
    [
        baseline_metrics,
        lightgbm_metrics,
        huber_metrics,
    ],
    index=[
        "Baseline",
        "Current L1",
        "Huber",
    ],
)

comparison

In [ ]:
FINAL_MAGNITUDE_MODEL = "lightgbm_log_magnitude_large_tree"

final_magnitude_decision = {
    "selected_model": FINAL_MAGNITUDE_MODEL,
    "magnitude_score": float(large_tree_metrics["magnitude_score"]),
    "mae": float(large_tree_metrics["mae"]),
    "rmse": float(large_tree_metrics["rmse"]),
    "rank_ic": float(large_tree_metrics["rank_ic"]),
    "rank_ic_t": float(large_tree_metrics["rank_ic_t"]),
    "r2_vs_vol": float(large_tree_metrics["r2_vs_vol"]),
    "test_used_for_selection": False,
}

final_magnitude_decision